In [8]:
import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets

from  sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score

In [9]:
from pathlib  import Path
data_path = Path('../data/raw')

files = {'grid':'trentino-grid.geojson',
         'adm_reg':'administrative_regions_Trentino.json',
        'weather':'meteotrentino-weather-station-data.json',
        'precip':'precipitation-trentino.csv',
        'precip-avail':'precipitation-trentino-data-availability.csv',
        'SET-1':'SET-nov-2013.csv',
        'SET-2':'SET-dec-2013.csv',
        'SET-lines':'line.csv',
        'twitter':'social-pulse-trentino.gejson'}

In [17]:
power_names = ['lineset', 'date_time', 'power']
pow_nov_df = pd.read_csv(data_path / files['SET-1'],names=power_names)
pow_dec_df = pd.read_csv(data_path / files['SET-2'],names=power_names)
pow_pos_df = pd.read_csv(data_path / files['SET-lines'])

In [13]:
# costruisco e concateno i dataframe con la potenza
lines_df = pd.concat([pow_nov_df, pow_dec_df], ignore_index=True)
lines_df = lines_df.sort_values(['lineset', 'date_time'])

date_time = pd.to_datetime(lines_df['date_time'])
date = date_time.dt.date
hour = date_time.dt.hour
lines_df['date'] = date
lines_df['hour'] = hour
lines_df = lines_df.drop(columns=['date_time'])

lines_df.head()

,lineset,power,date,hour
0,DG1000420,37.439999,2013-11-01,0
1,DG1000420,37.439999,2013-11-01,0
2,DG1000420,36.000000,2013-11-01,0
3,DG1000420,35.279999,2013-11-01,0
4,DG1000420,35.279999,2013-11-01,0


In [14]:
# dato che gli inquinanti atmosferici sono misurati su base oraria, medio anche la potenza su ciascuna ora. Per semplicità chiamo potenza ad una data ora la
# la potenza emessa dalle 00:00 di quell'ora alle 59:59 di quell'ora
lines_df = lines_df.groupby(['lineset', 'date', 'hour'])['power'].mean().reset_index()
lines_df.head()

,lineset,date,hour,power
0,DG1000420,2013-11-01,0,35.939999
1,DG1000420,2013-11-01,1,33.779998
2,DG1000420,2013-11-01,2,31.889998
3,DG1000420,2013-11-01,3,31.979998
4,DG1000420,2013-11-01,4,31.349998


In [62]:
tot_ub_df

,date,hour,tot_ub
lineset,,,
DG1000420,2013-11-01,0,2133
DG1000421,2013-11-01,0,1424
DG1000422,2013-11-01,0,3423
DG1000423,2013-11-01,0,656
DG1000425,2013-11-01,0,2013
...,...,...,...
DG1056621,2013-11-01,0,49
DG1056623,2013-11-01,0,149
DG1056624,2013-11-01,0,1087


In [ ]:
# ogni linea elettrica distribuisce corrente elettrica su diverse celle. Mergiamo il df con la potenza per linea, con quello con il numero di ubicazioni 
# per cella a cui ogni linea fornisce energia. Assumendo un consumo uniforme per ubicazione, cerchiamo poi quanta potenza venga effettivamente consumata in
# ciascuna cella.

el_power_df = pd.merge(pow_pos_df,lines_df,left_on='LINESET',right_on='lineset',how='right')

# calcolo il totale di ubicazioni per ogni linea
tot_ub_df = el_power_df.groupby(['lineset', 'date', 'hour'])['NR_UBICAZIONI'].sum().reset_index().drop_duplicates(subset='lineset')
tot_ub_df = tot_ub_df.rename(columns={'NR_UBICAZIONI': 'tot_ub'})
el_power_df['tot_ub'] = el_power_df['lineset'].map(tot_ub_df.set_index('lineset')['tot_ub'])

el_power_df['power_square'] = el_power_df['power'] * el_power_df['NR_UBICAZIONI'] / el_power_df['tot_ub']
el_power_df = el_power_df.rename(columns={'SQUAREID': 'squareid'})
el_power_df.head()

,squareid,LINESET,NR_UBICAZIONI,lineset,date,hour,power,tot_ub,power_square
0,4037,DG1000420,2,DG1000420,2013-11-01,0,35.939999,2133,0.033699
1,4154,DG1000420,13,DG1000420,2013-11-01,0,35.939999,2133,0.219044
2,4155,DG1000420,20,DG1000420,2013-11-01,0,35.939999,2133,0.336990
3,4156,DG1000420,2,DG1000420,2013-11-01,0,35.939999,2133,0.033699
4,4269,DG1000420,1,DG1000420,2013-11-01,0,35.939999,2133,0.016850
...,...,...,...,...,...,...,...,...,...
3705379,8502,DG1056626,1,DG1056626,2013-12-31,23,77.975000,1667,0.046776
3705380,8503,DG1056626,4,DG1056626,2013-12-31,23,77.975000,1667,0.187103
3705381,8621,DG1056626,1,DG1056626,2013-12-31,23,77.975000,1667,0.046776
3705382,8972,DG1056626,25,DG1056626,2013-12-31,23,77.975000,1667,1.169391


In [69]:
# quello che interessa a me è di trovare la potenza media oraria per quandrato. Quindi raggruppo per quadrato e sommo
sq_power_df = el_power_df.groupby(['squareid', 'date', 'hour'])['power_square'].sum().reset_index()
sq_power_df.head()

,squareid,date,hour,power_square
0,155,2013-11-01,0,0.090551
1,155,2013-11-01,1,0.075594
2,155,2013-11-01,2,0.077797
3,155,2013-11-01,3,0.072870
4,155,2013-11-01,4,0.073739


In [72]:
# adesso voglio finalmente aggiungere la geometria facendo riferimento alla griglia

grid_df = gpd.read_file(data_path / files['grid'])
sq_power_df['geometry'] = sq_power_df['squareid'].map(grid_df.set_index('cellId')['geometry'])
sq_power_df.head()

,squareid,date,hour,power_square,geometry
0,155,2013-11-01,0,0.090551,"POLYGON ((10.91493 45.691, 10.92777 45.69079, ..."
1,155,2013-11-01,1,0.075594,"POLYGON ((10.91493 45.691, 10.92777 45.69079, ..."
2,155,2013-11-01,2,0.077797,"POLYGON ((10.91493 45.691, 10.92777 45.69079, ..."
3,155,2013-11-01,3,0.072870,"POLYGON ((10.91493 45.691, 10.92777 45.69079, ..."
4,155,2013-11-01,4,0.073739,"POLYGON ((10.91493 45.691, 10.92777 45.69079, ..."


In [74]:
# vogliamo quindi capire quanto riescano effettivamente a muoversi gli inquinanti in atmosfera in modo da poter assegnare a ciascuna stazione un raggio entro
# cui l'eventuale emissione di inquinanti può essere effettivamente rilevata dalla stazione stessa.
# leggendo online, sembrano ci siano diversi modelli che possono descrivere questa dispersione degli inquinanti nella bassa atmosfera
# https://it.wikipedia.org/wiki/Modelli_di_dispersione_in_atmosfera
# studiare approfonditamente il fenomeno va ben oltre gli scopi di questo progetto. Assumiamo una descrizione approssimata e semplice della dispersione.

# da quanto ho capito, se viene trascurata la presenza del vento, i due principali vettori della dispersione sono il movimento browniano delle particelle in aria
# ed i moti turbolenti che si instaurano in presenza di gradienti termici. In generale sembra che il fenomeno browniano sia sostanzialmente trascurabile, portando
# a movimenti di solo qualche metro al giorno. Quello che fa la differenza sono invece i moti turbolenti, che possono trasportare l'inquinante anche a diverse
# centinaia di metri nel giro di qualche ora. 
# La distanza percorsa dalle particelle in questo secondo caso dipende dal coefficiente di diffusione turbolenta.

# creiamo tre colonne ed in ogni colonna assumiamo un andamento diverso degli inquinanti: 
# * colonna 1: istantanea, solo la casella in cui si trova la stazione
# * colonna 2: dopo un'ora, tutte le caselle entro 100 metri
# * colonna 3: dopo un giorno, tutte le caselle entro 400 metri